# LORE-SA Tutorial Adapted to ProPublica Recidivism Data

This notebook adapts the original LORE-SA tutorial from the Adult dataset to the provided `propublica-recidivism_crim.csv` dataset.

**Project focus:** audit whether counterfactual explanations/recourse suggestions respect data governance rules.  
We do not rewrite the LORE-SA algorithm. We run it, then check whether its generated counterfactual rules suggest changes to protected or non-actionable attributes.


## Explanation 1 — What we are auditing

LORE produces two types of explanations:

1. **Factual rule:** why the black-box model predicted the current class.
2. **Counterfactual rule:** what would need to change for the prediction to become different.

For a governance audit, the key question is:

> Does the counterfactual recourse suggest changing protected or impossible-to-change attributes such as sex, race, or age?

If yes, we mark the explanation as **non-compliant**.


Loaded, cleaned, and transformed the ProPublica dataset. Created the age variable and selected relevant features.
Obtained a clean dataset of 13,639 records and 9 attributes ready for LORE-SA.

In [1]:
from lore_sa.dataset import TabularDataset
import pandas as pd
import numpy as np
from pathlib import Path

# Use the CSV provided with this project.
# If you run the notebook locally, keep this CSV in the same folder as the notebook.
CSV_PATH = Path("propublica-recidivism_crim.csv")
if not CSV_PATH.exists():
    # Fallback for environments like Google Colab / ChatGPT sandbox
    CSV_PATH = Path("/mnt/data/propublica-recidivism_crim.csv")

raw_df = pd.read_csv(CSV_PATH)

# Keep only the COMPAS-style "Risk of Recidivism" rows.
df = raw_df[
    (raw_df["DisplayText"] == "Risk of Recidivism") &
    (raw_df["IsCompleted"] == 1) &
    (raw_df["IsDeleted"] == 0)
].copy()

# Convert dates and create age at screening.
df["DateOfBirth"] = pd.to_datetime(df["DateOfBirth"], errors="coerce")
df["Screening_Date"] = pd.to_datetime(df["Screening_Date"], errors="coerce")
df["age"] = ((df["Screening_Date"] - df["DateOfBirth"]).dt.days / 365.25).round(0)

# Build a clean tabular dataset for LORE.
# Target: ScoreText = Low / Medium / High risk label.
lore_df = df[[
    "age",
    "Sex_Code_Text",
    "Ethnic_Code_Text",
    "LegalStatus",
    "CustodyStatus",
    "MaritalStatus",
    "AssessmentReason",
    "Language",
    "ScoreText"
]].copy()

lore_df.columns = [
    "age",
    "sex",
    "race",
    "legal_status",
    "custody_status",
    "marital_status",
    "assessment_reason",
    "language",
    "class"
]

# Basic cleaning
lore_df = lore_df.dropna()
lore_df = lore_df[(lore_df["age"] >= 18) & (lore_df["age"] <= 100)]
lore_df["age"] = lore_df["age"].astype(int)

# Save the prepared CSV because TabularDataset.from_csv expects a file path.
PREPARED_PATH = Path("propublica_lore_ready.csv")
lore_df.to_csv(PREPARED_PATH, index=False)

print("Prepared dataset shape:", lore_df.shape)
display(lore_df.head())
print(lore_df["class"].value_counts())

dataset = TabularDataset.from_csv(str(PREPARED_PATH), class_name="class")
dataset.df.dropna(inplace=True)
dataset.update_descriptor()
dataset.df.head()


C:\Users\user\AppData\Local\Temp\ipykernel_20740\245839633.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DateOfBirth"] = pd.to_datetime(df["DateOfBirth"], errors="coerce")


Prepared dataset shape: (13639, 9)


C:\Users\user\AppData\Local\Temp\ipykernel_20740\245839633.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Screening_Date"] = pd.to_datetime(df["Screening_Date"], errors="coerce")


,age,sex,race,legal_status,custody_status,marital_status,assessment_reason,language,class
1,20,Male,Caucasian,Pretrial,Jail Inmate,Single,Intake,English,Low
4,28,Male,Caucasian,Pretrial,Jail Inmate,Married,Intake,English,Low
7,18,Male,African-American,Pretrial,Jail Inmate,Single,Intake,English,High
10,18,Female,African-American,Pretrial,Jail Inmate,Significant Other,Intake,English,Medium
13,28,Female,African-American,Pretrial,Jail Inmate,Single,Intake,English,Low


class
Low       6445
Medium    4048
High      3146
Name: count, dtype: int64


,age,sex,race,legal_status,custody_status,marital_status,assessment_reason,language,class
0,20,Male,Caucasian,Pretrial,Jail Inmate,Single,Intake,English,Low
1,28,Male,Caucasian,Pretrial,Jail Inmate,Married,Intake,English,Low
2,18,Male,African-American,Pretrial,Jail Inmate,Single,Intake,English,High
3,18,Female,African-American,Pretrial,Jail Inmate,Significant Other,Intake,English,Medium
4,28,Female,African-American,Pretrial,Jail Inmate,Single,Intake,English,Low


In [2]:
dataset.descriptor.keys()

dict_keys(['numeric', 'categorical', 'ordinal', 'target'])

Created the dataset descriptor used by LORE-SA.
LORE identified numeric, categorical, ordinal, and target attributes correctly.

## Train a black-box model

This is the model that LORE will explain.  
The model predicts the risk label `class` from the ProPublica features.


Introduced the black-box model section.
Prepared for model training.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from lore_sa.bbox import sklearn_classifier_bbox


def train_model(dataset: TabularDataset):
    feature_columns = [c for c in dataset.df.columns if c != "class"]
    X = dataset.df[feature_columns]
    y = dataset.df["class"]

    numeric_features = ["age"]
    categorical_features = [c for c in feature_columns if c not in numeric_features]

    numeric_idx = [feature_columns.index(c) for c in numeric_features]
    categorical_idx = [feature_columns.index(c) for c in categorical_features]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_idx),
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_idx)
        ]
    )

    model = make_pipeline(
        preprocessor,
        RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X.values,
        y.values,
        test_size=0.3,
        random_state=42,
        stratify=y.values
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))

    return sklearn_classifier_bbox.sklearnBBox(model)


bbox = train_model(dataset)


Accuracy: 0.46603128054740955
              precision    recall  f1-score   support

        High       0.35      0.55      0.43       944
         Low       0.65      0.52      0.57      1934
      Medium       0.37      0.32      0.34      1214

    accuracy                           0.47      4092
   macro avg       0.45      0.46      0.45      4092
weighted avg       0.50      0.47      0.47      4092



Trained a Random Forest classifier using age and categorical attributes.
Achieved about 46.6% accuracy and created the black-box model (bbox).

## Generate a LORE explanation

This section is the original tutorial logic: create the explainer and explain one selected person/row.


Created the LORE explainer using the trained model and dataset.
LORE-SA became ready to generate explanations.

In [4]:
from lore_sa.lore import TabularRandomGeneratorLore

tabularLore = TabularRandomGeneratorLore(bbox, dataset)


Created the LORE explainer using the trained model and dataset.
LORE-SA became ready to generate explanations.

In [5]:
num_row = 10

x = dataset.df.iloc[num_row][:-1]  # exclude target feature
print("Instance to explain:")
display(x.to_frame().T)

explanation = tabularLore.explain(x)
print(explanation)


Instance to explain:


,age,sex,race,legal_status,custody_status,marital_status,assessment_reason,language
10,34,Female,Caucasian,Pretrial,Jail Inmate,Married,Intake,English


{'rule': {'premises': [{'attr': 'age', 'val': 29.796560287475586, 'op': '>'}, {'attr': 'sex', 'val': 'Male', 'op': '!='}, {'attr': 'custody_status', 'val': 'Prison Inmate', 'op': '!='}, {'attr': 'marital_status', 'val': 'Significant Other', 'op': '!='}], 'consequence': {'attr': 'class', 'val': 'Low', 'op': '='}}, 'counterfactuals': [{'premises': [{'attr': 'age', 'val': 20.12177848815918, 'op': '<='}, {'attr': 'race', 'val': 'African-Am', 'op': '!='}, {'attr': 'legal_status', 'val': 'Deferred Sentencing', 'op': '!='}, {'attr': 'marital_status', 'val': 'Divorced', 'op': '='}], 'consequence': {'attr': 'class', 'val': 'Medium', 'op': '='}}, {'premises': [{'attr': 'age', 'val': 24.71651554107666, 'op': '<='}, {'attr': 'age', 'val': 20.12177848815918, 'op': '>'}, {'attr': 'race', 'val': 'African-Am', 'op': '!='}, {'attr': 'custody_status', 'val': 'Pretrial Defendant', 'op': '='}, {'attr': 'legal_status', 'val': 'Other', 'op': '!='}, {'attr': 'legal_status', 'val': 'Probation Violator', 'op':

Tested encoding and decoding mechanisms.
Verified that data transformation works correctly before explanation generation.

In [6]:
from lore_sa.encoder_decoder import ColumnTransformerEnc

tabular_enc = ColumnTransformerEnc(dataset.descriptor)

ref_value = dataset.df.iloc[0].values[:-1]
encoded = tabular_enc.encode([ref_value])
decoded = tabular_enc.decode(encoded)

print(f"Original value: {ref_value}")
print(f"Encoded value: {encoded}")
print(f"Decoded value: {decoded}")


Original value: [20 'Male' 'Caucasian' 'Pretrial' 'Jail Inmate' 'Single' 'Intake'
 'English']
Encoded value: [[20 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 1 0]]
Decoded value: [[20 'Male' 'Caucasian' 'Pretrial' 'Jail Inmate' 'Single' 'Intake'
  'English']]


10: Generated 100 synthetic neighboring instances around the selected person.
Created a local neighborhood used by LORE to understand model behavior.

In [7]:
from lore_sa.neighgen import RandomGenerator

num_row = 10

x = dataset.df.iloc[num_row][:-1]
z = tabular_enc.encode([x.values])[0]  # remove the class feature from the input instance

gen = RandomGenerator(bbox=bbox, dataset=dataset, encoder=tabular_enc, ocr=0.1)
neighbour = gen.generate(z, 100, dataset.descriptor, tabular_enc)

print("Neighborhood shape:", neighbour.shape)
print(neighbour[:5])


Neighborhood shape: (100, 35)
[[34 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0]
 [34 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0]
 [37.014180874938475 1 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
  0 0 0 0 0 1 1 0]
 [37.014180874938475 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
  0 0 0 0 0 1 0 1]
 [33.140298317308215 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
  0 0 0 0 0 1 0 1]]


In [8]:
from lore_sa.surrogate import DecisionTreeSurrogate

# Decode the neighborhood and label it with the black-box model.
neighb_train_X = tabular_enc.decode(neighbour)
neighb_train_y = bbox.predict(neighb_train_X)

# Encode the target class for the surrogate decision tree.
neighb_train_yz = tabular_enc.encode_target_class(neighb_train_y.reshape(-1, 1)).squeeze()

dt = DecisionTreeSurrogate()
dt.train(neighbour, neighb_train_yz)


DecisionTreeClassifier(class_weight='balanced', random_state=42)

In [9]:
num_row = 10

x = dataset.df.iloc[num_row][:-1]
z = tabular_enc.encode([x.values])[0]

rule = dt.get_rule(z, tabular_enc)
print("Factual rule")
print(rule)

crules, deltas = dt.get_counterfactual_rules(z, neighbour, neighb_train_yz, tabular_enc)

print("\nCounterfactual rules")
for i, c in enumerate(crules, 1):
    print(f"{i}. {c}")


Factual rule
premises:
age > 33.080392837524414 
consequence: class = Low

Counterfactual rules
1. premises:
age <= 23.789048194885254
marital_status=Widowed = False
race=Asian = False
legal_status=Pretrial = False 
consequence: class = Medium
2. premises:
age <= 33.080392837524414
age > 31.813100814819336
marital_status=Widowed = False
legal_status=Parole Violator = True
race=African-American = True 
consequence: class = High
3. premises:
age <= 28.302794456481934
age > 23.789048194885254
marital_status=Widowed = False
custody_status=Pretrial Defendant = False 
consequence: class = High
4. premises:
age <= 31.813100814819336
age > 28.302794456481934
marital_status=Widowed = False
legal_status=Parole Violator = True
race=African-American = True 
consequence: class = High


## Explanation 2 — Governance policy for ProPublica data

For this dataset, many variables are **not actionable**. A person cannot ethically be told to change their race, sex, age, legal status, custody status, language, or marital status in order to receive a different risk prediction.

So the audit policy is:

- **Protected attributes:** `sex`, `race`, `age`
- **Other non-actionable attributes:** `legal_status`, `custody_status`, `marital_status`, `assessment_reason`, `language`
- A counterfactual is **compliant** only if it avoids all protected and non-actionable attributes.


In [10]:
# Governance policy

protected_attributes = [
    "sex",
    "race",
    "age"
]

non_actionable_attributes = [
    "legal_status",
    "custody_status",
    "marital_status",
    "assessment_reason",
    "language"
]

forbidden_attributes = protected_attributes + non_actionable_attributes

print("Protected attributes:", protected_attributes)
print("Non-actionable attributes:", non_actionable_attributes)


Protected attributes: ['sex', 'race', 'age']
Non-actionable attributes: ['legal_status', 'custody_status', 'marital_status', 'assessment_reason', 'language']


In [11]:
def audit_counterfactual_rule(counterfactual_rule, forbidden_attributes):
    """
    Return whether a LORE counterfactual rule is governance-compliant.
    A rule is non-compliant if it mentions a protected or non-actionable attribute.
    """
    rule_text = str(counterfactual_rule).lower()

    used_forbidden = [
        attr for attr in forbidden_attributes
        if attr.lower() in rule_text
    ]

    return {
        "counterfactual": str(counterfactual_rule),
        "used_forbidden_attributes": used_forbidden,
        "status": "Compliant" if len(used_forbidden) == 0 else "Non-compliant"
    }


audit_results = [
    audit_counterfactual_rule(c, forbidden_attributes)
    for c in crules
]

audit_df = pd.DataFrame(audit_results)
display(audit_df)

print("Governance audit summary:")
print(audit_df["status"].value_counts())


,counterfactual,used_forbidden_attributes,status
0,premises:\nage <= 23.789048194885254\nmarital_...,"[race, age, legal_status, marital_status]",Non-compliant
1,premises:\nage <= 33.080392837524414\nage > 31...,"[race, age, legal_status, marital_status]",Non-compliant
2,premises:\nage <= 28.302794456481934\nage > 23...,"[age, custody_status, marital_status]",Non-compliant
3,premises:\nage <= 31.813100814819336\nage > 28...,"[race, age, legal_status, marital_status]",Non-compliant


Governance audit summary:
status
Non-compliant    4
Name: count, dtype: int64


In [12]:
# Separate acceptable and rejected recourse suggestions.

compliant_recourse = audit_df[audit_df["status"] == "Compliant"]
rejected_recourse = audit_df[audit_df["status"] == "Non-compliant"]

print("Compliant recourse suggestions:")
if len(compliant_recourse) == 0:
    print("No compliant recourse found for this instance.")
else:
    display(compliant_recourse)

print("\nRejected recourse suggestions:")
if len(rejected_recourse) == 0:
    print("No rejected recourse.")
else:
    display(rejected_recourse)


Compliant recourse suggestions:
No compliant recourse found for this instance.

Rejected recourse suggestions:


,counterfactual,used_forbidden_attributes,status
0,premises:\nage <= 23.789048194885254\nmarital_...,"[race, age, legal_status, marital_status]",Non-compliant
1,premises:\nage <= 33.080392837524414\nage > 31...,"[race, age, legal_status, marital_status]",Non-compliant
2,premises:\nage <= 28.302794456481934\nage > 23...,"[age, custody_status, marital_status]",Non-compliant
3,premises:\nage <= 31.813100814819336\nage > 28...,"[race, age, legal_status, marital_status]",Non-compliant


## Comparative experiment: original LORE vs governance-filtered recourse

The LORE algorithm still generates its normal counterfactual rules.  
The governance layer does not change the algorithm; it evaluates the results.

- **Original LORE output:** all generated counterfactual rules.
- **Governance-filtered output:** only counterfactual rules that do not use protected or non-actionable attributes.

This gives the comparison needed for the project proposal.


In [13]:
def explain_and_audit_instance(row_index, n_neighbours=100):
    x = dataset.df.iloc[row_index][:-1]
    z = tabular_enc.encode([x.values])[0]

    neighbour = gen.generate(z, n_neighbours, dataset.descriptor, tabular_enc)
    neighb_train_X = tabular_enc.decode(neighbour)
    neighb_train_y = bbox.predict(neighb_train_X)
    neighb_train_yz = tabular_enc.encode_target_class(neighb_train_y.reshape(-1, 1)).squeeze()

    dt = DecisionTreeSurrogate()
    dt.train(neighbour, neighb_train_yz)

    rule = dt.get_rule(z, tabular_enc)
    crules, deltas = dt.get_counterfactual_rules(z, neighbour, neighb_train_yz, tabular_enc)

    audit_results = [audit_counterfactual_rule(c, forbidden_attributes) for c in crules]
    audit_df = pd.DataFrame(audit_results)

    return {
        "row_index": row_index,
        "prediction": bbox.predict([x.values])[0],
        "factual_rule": str(rule),
        "n_counterfactuals": len(crules),
        "n_compliant": int((audit_df["status"] == "Compliant").sum()) if len(audit_df) else 0,
        "n_non_compliant": int((audit_df["status"] == "Non-compliant").sum()) if len(audit_df) else 0,
        "counterfactuals": audit_df
    }


rows_to_audit = [10, 25, 50, 75, 100]
summary = []
detailed_results = {}

for row in rows_to_audit:
    result = explain_and_audit_instance(row)
    detailed_results[row] = result

    summary.append({
        "row_index": result["row_index"],
        "prediction": result["prediction"],
        "n_counterfactuals_original_lore": result["n_counterfactuals"],
        "n_counterfactuals_after_governance_filter": result["n_compliant"],
        "n_rejected_by_governance": result["n_non_compliant"]
    })

summary_df = pd.DataFrame(summary)
display(summary_df)


,row_index,prediction,n_counterfactuals_original_lore,n_counterfactuals_after_governance_filter,n_rejected_by_governance
0,10,Low,1,0,1
1,25,High,2,0,2
2,50,Low,2,0,2
3,75,Low,3,0,3
4,100,High,1,0,1


## Final interpretation template

Use this in your report:

> We adapted the LORE-SA tutorial to the ProPublica recidivism dataset. The black-box model predicts the `ScoreText` risk category. LORE generates factual and counterfactual rules for individual predictions. We then apply a governance audit that rejects counterfactual recourse suggestions involving protected attributes (`sex`, `race`, `age`) or other non-actionable variables (`legal_status`, `custody_status`, `marital_status`, `assessment_reason`, `language`). The comparative experiment reports how many original LORE counterfactuals remain valid after governance filtering.
